# Notebook 1: Historical Simulation VaR
**Author:** Niraj Neupane | github.com/nirajneupane17
**Assets:** SPY 25% / QQQ 25% / TLT 20% / GLD 15% / EFA 15%

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 observations | 2015-01-01 to 2024-12-31


## 1. Portfolio Summary

In [2]:
print('Portfolio Statistics:')
print(f'  Daily mean   : {port.mean():.6f}')
print(f'  Daily std    : {port.std():.6f}')
print(f'  Annual vol   : {port.std()*np.sqrt(252)*100:.2f}%')
print(f'  Min return   : {port.min():.4f}')
print(f'  Max return   : {port.max():.4f}')
print(f'  Skewness     : {port.skew():.4f}')
print(f'  Kurtosis     : {port.kurtosis():.4f}')

Portfolio Statistics:
  Daily mean   : -0.000142
  Daily std    : 0.007380
  Annual vol   : 11.72%
  Min return   : -0.0796
  Max return   : 0.0466
  Skewness     : -0.8391
  Kurtosis     : 8.2896


## 2. Rolling Historical VaR

In [3]:
from var_models import historical_var
var_rolling = historical_var(port, window=252, confidence_levels=[0.95, 0.99, 0.995])
print('Rolling VaR (last 5 rows):')
print(var_rolling.tail().round(4))

Rolling VaR (last 5 rows):
            VaR_95pct  VaR_99pct
Date                            
2024-12-25     0.0098     0.0134
2024-12-26     0.0098     0.0134
2024-12-27     0.0098     0.0134
2024-12-30     0.0098     0.0134
2024-12-31     0.0098     0.0134


In [4]:
fig, axes = plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
ax1.fill_between(port.index,port,0,where=port<0,color='#E24B4A',alpha=0.5,label='Losses')
ax1.fill_between(port.index,port,0,where=port>=0,color='#1D9E75',alpha=0.4,label='Gains')
ax1.set_title('Portfolio Daily Returns'); ax1.set_ylabel('Return'); ax1.legend(fontsize=9)
ax2=axes[1]
for col,c,lbl in zip(var_rolling.columns,['#185FA5','#E24B4A','#7B1FA2'],['95%','99%','99.5%']):
    ax2.plot(var_rolling.index,var_rolling[col],color=c,linewidth=1.3,label=f'{lbl} VaR')
ax2.set_title('Rolling 252-Day Historical VaR'); ax2.set_ylabel('VaR'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/01_historical_var_rolling.png',dpi=150,bbox_inches='tight')
plt.show(); print('Chart saved.')

Chart saved.


## 3. Full-Period VaR Summary

In [5]:
from var_models import var_summary_table
summary = var_summary_table(port)
print('VaR Summary (full period):')
print(summary.round(4))
summary.to_csv('../results/07_var_summary.csv')

VaR Summary (full period):
            daily_VaR  10day_VaR  annual_VaR
confidence                                  
90%            0.0089     0.0281      0.1408
95%            0.0115     0.0364      0.1826
99%            0.0168     0.0530      0.2663
99%            0.0212     0.0669      0.3358


## 4. Return Distribution

In [6]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.hist(port,bins=80,color='#185FA5',alpha=0.7,edgecolor='white',linewidth=0.3,density=True)
for cl,c,lbl in zip([0.95,0.99,0.995],['#F57C00','#E24B4A','#7B1FA2'],['95%','99%','99.5%']):
    v=np.percentile(port,(1-cl)*100)
    ax1.axvline(v,color=c,linewidth=1.8,linestyle='--',label=f'{lbl} VaR={abs(v):.3f}')
x=np.linspace(port.min(),port.max(),300)
ax1.plot(x,stats.norm.pdf(x,port.mean(),port.std()),'k--',linewidth=1.2,alpha=0.6,label='Normal fit')
ax1.set_title('Return Distribution with VaR Thresholds'); ax1.set_xlabel('Return'); ax1.legend(fontsize=8)
ax2=axes[1]
(osm,osr),(sl,ic,_)=stats.probplot(port,dist='norm')
ax2.scatter(osm,osr,s=5,color='#185FA5',alpha=0.5)
ax2.plot(osm,sl*np.array(osm)+ic,'r--',linewidth=1.5,label='Normal reference')
ax2.set_title('Q-Q Plot: Empirical vs Normal'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/02_return_distribution.png',dpi=150,bbox_inches='tight')
plt.show(); print('Chart saved.')

Chart saved.


## 5. Key Findings

In [7]:
print('='*55)
print('  HISTORICAL VAR — KEY FINDINGS')
print('='*55)
print(f'  Skewness     : {port.skew():.4f}')
print(f'  Kurtosis     : {port.kurtosis():.4f} (fat tails confirmed)')
print(f'  95% Daily VaR: {abs(np.percentile(port,5)):.4f}')
print(f'  99% Daily VaR: {abs(np.percentile(port,1)):.4f}')
print(f'  99.5% Daily  : {abs(np.percentile(port,0.5)):.4f}')
print()
print('  Fat tails present — Student-t VaR preferred over Normal.')
print('='*55)

  HISTORICAL VAR — KEY FINDINGS
  Skewness     : -0.8391
  Kurtosis     : 8.2896 (fat tails confirmed)
  95% Daily VaR: 0.0115
  99% Daily VaR: 0.0168
  99.5% Daily  : 0.0212

  Fat tails present — Student-t VaR preferred over Normal.
